# IRI Standards Agent — Production v1.0 Test
Validates agent against FundTransfer v1.2.0 spec. Expected benchmark: 87/100.

In [0]:
%pip install strands-agents strands-agents-tools openai pyyaml databricks-sdk --quiet
dbutils.library.restartPython()

In [0]:
import sys
sys.path.insert(0, '/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent')

from agent.agent_prod import create_agent
agent = create_agent()

# Patch: Databricks endpoints don't always return token usage metrics,
# which causes strands event loop to crash with NoneType += error
_original_update = agent.event_loop_metrics.update_usage
def _safe_update(usage):
    try:
        _original_update(usage)
    except TypeError:
        pass  # Skip metrics update when usage data is None
agent.event_loop_metrics.update_usage = _safe_update
print("[patch] Applied metrics safety patch for Databricks endpoint")

In [0]:
spec_path = '/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/draft-api-specs/FundTransfer_v1.2.0.yml'
with open(spec_path) as f:
    spec_yaml = f.read()
print(f"Loaded FundTransfer spec: {len(spec_yaml)} chars, {spec_yaml.count(chr(10))+1} lines")

In [0]:
review_prompt = f"""Review this IRI Digital-First OpenAPI 3.1 YAML specification.

This is a revision of a previously reviewed spec. Focus on structural, style guide, and cross-spec consistency issues.
Please check for cross-spec consistency against published IRI specs.

After your review, provide:
1. The full governance scorecard with category scores
2. A list of ALL findings (Critical, Moderate, Minor) with evidence

Here is the spec:

```yaml
{spec_yaml}
```"""

result = agent(review_prompt)
review_output = str(result)
print(review_output[:3000])

In [0]:
fix_prompt = f"""Based on your review above, please generate the CORRECTED version of the FundTransfer spec.

Apply ALL Critical and Moderate fixes. For Minor issues, fix those that are straightforward (missing descriptions, formatting).

Output the COMPLETE corrected YAML file — do not truncate or summarize. Include every endpoint, schema, and component."""

fix_result = agent(fix_prompt)
fix_output = str(fix_result)
print(f"Fix response length: {len(fix_output)} chars")
print(fix_output[:2000])

In [0]:
import re

# Extract YAML block from response
yaml_match = re.search(r'```ya?ml\n(.*?)```', fix_output, re.DOTALL)
if yaml_match:
    corrected_yaml = yaml_match.group(1)
else:
    # Try to find raw YAML (starts with openapi:)
    yaml_start = fix_output.find('openapi:')
    if yaml_start >= 0:
        corrected_yaml = fix_output[yaml_start:]
    else:
        corrected_yaml = fix_output
        print("⚠️ Could not isolate YAML block — saving full output")

output_path = '/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/output/FundTransfer_v1.2.0_prod_corrected.yml'
with open(output_path, 'w') as f:
    f.write(corrected_yaml)
print(f"✅ Saved corrected spec: {len(corrected_yaml)} chars, {corrected_yaml.count(chr(10))+1} lines")
print(f"   Path: {output_path}")

In [0]:
rescore_prompt = f"""Now re-review the CORRECTED spec below and provide the final governance scorecard.

Give me:
1. Overall score and PASS/CONDITIONAL/FAIL status
2. Category breakdown (A through E with points)
3. Any remaining findings

```yaml
{corrected_yaml}
```"""

final_result = agent(rescore_prompt)
final_output = str(final_result)
print("=" * 60)
print("FINAL SCORECARD — FundTransfer v1.2.0 (Corrected)")
print("=" * 60)
print(final_output)

In [0]:
print("=" * 60)
print("PRODUCTION AGENT v1.0 — TEST RESULTS SUMMARY")
print("=" * 60)
print(f"\nSpec: FundTransfer v1.2.0")
print(f"Agent: agent_prod.py (calibrated prompt + 2 fetch tools)")
print(f"Model: databricks-claude-opus-4-7")
print(f"\nOriginal Review:")
print(f"  Output length: {len(review_output)} chars")
print(f"\nCorrected Spec:")
print(f"  Output path: output/FundTransfer_v1.2.0_prod_corrected.yml")
print(f"  Size: {len(corrected_yaml)} chars")
print(f"\nFinal Scorecard:")
print(f"  Output length: {len(final_output)} chars")
print(f"\n{'=' * 60}")